<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Tune hyperparameters

**[Tune hyperparameters](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/)** · Machine Learning: From Problem to Reliable Model · Module 6, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** tell parameters from hyperparameters, choose hyperparameters with cross-validation on the train set (grid search and random search), decide with the one-standard-error rule, and see why the test set must never help to choose.

| | |
|---|---|
| **Time** | About 65 minutes. The searches take about 15 seconds each. |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Cross-validation, from [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/#cross-validation). Underfitting and overfitting, from [Control complexity](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/). Average precision, from [The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/#average-precision). |
| **You do not need** | The local project. The setup below downloads the data and the final `ticket_model.py`. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M06-L02-tune-hyperparameters/tune-hyperparameters-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. `ticket_model.py` is the final 11-feature version, from [Select useful features](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/#change-ticket_modelpy).

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: the experiment log

The next cell defines two helpers for your **experiment log**, the file `experiment_log.csv` in `ticket-model/`. You start the log in [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log).

- `log_experiment(id, change, cv_ap, cv_std, valid_ap, decision, reason)` adds one row, with today's date. If the id is already in the log, it replaces that row, so you can run a cell again.
- `show_log()` shows the log as a table.

It also writes the 14 rows that you added in the earlier lessons (E01 to E14), so this notebook works without them. If you keep your own log in the local project, skip the last line of the cell.

> **Check.** You should see the log with 14 rows, from E01 to E14.

In [ ]:
# The experiment log: one row per experiment, in experiment_log.csv.
import datetime
from pathlib import Path

LOG_FILE = Path("experiment_log.csv")
LOG_COLUMNS = ["id", "date", "change", "cv_ap", "cv_std", "valid_ap", "decision", "reason"]


def log_experiment(id, change, cv_ap, cv_std, valid_ap=None, decision="", reason=""):
    """Add one row to the log (or replace the row with the same id)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}) if LOG_FILE.exists() else pd.DataFrame(columns=LOG_COLUMNS)
    row = {"id": id, "date": datetime.date.today().isoformat(), "change": change,
           "cv_ap": round(float(cv_ap), 3), "cv_std": round(float(cv_std), 3),
           "valid_ap": None if valid_ap is None else round(float(valid_ap), 3),
           "decision": decision, "reason": reason}
    log = pd.concat([log[log["id"] != id], pd.DataFrame([row])], ignore_index=True)
    log.sort_values("id").to_csv(LOG_FILE, index=False)


def show_log(columns=("id", "change", "cv_ap", "cv_std", "valid_ap", "decision")):
    """The log as a table (without the date and the reason, unless you ask for them)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}).sort_values("id")
    with pd.option_context("display.max_colwidth", 60, "display.width", 200):
        print(log[list(columns)].fillna("").to_string(index=False))


# The rows of the earlier lessons (id, change, cv_ap, cv_std, valid_ap, decision, reason).
EARLIER_ROWS = [
    ('E01', 'Baseline: one score for every ticket (the train rate)', 0.117, 0.000, 0.155, 'baseline', 'The floor: any useful model must be above it.'),
    ('E02', "Baseline: the rule 'flag priority 1', as a 0/1 score", 0.178, 0.008, 0.228, 'baseline', 'What Larkfield does today: the model must beat it.'),
    ('E03', 'Logistic regression, the 10 C04 features', 0.327, 0.012, 0.353, 'keep', 'Far above both baselines; train AP 0.328, so it does not overfit.'),
    ('E04', "E03 with class_weight='balanced'", 0.323, 0.012, None, 'reject', 'Lower than E03 on all 5 folds; one more setting for no gain.'),
    ('E05', "Baseline: the rule 'flag priority 1 or team payment'", 0.172, 0.006, None, 'reject', 'Lower than the simpler rule E02.'),
    ('E06', 'Decision tree, max_depth=3', 0.234, 0.011, None, 'reject', 'Far below E03.'),
    ('E07', 'Decision tree, max_depth=6', 0.258, 0.010, None, 'reject', 'The best of the three trees, still far below E03.'),
    ('E08', 'Decision tree, no depth limit', 0.137, 0.004, None, 'reject', 'It memorises train (train AP 1.000) and fails on new tickets.'),
    ('E09', 'Decision tree, no depth limit, min_samples_leaf=100', 0.278, 0.017, None, 'reject', 'The best tree so far, still far below E03.'),
    ('E10', 'E03 with C=0.0001 (a very strong penalty)', 0.302, 0.012, None, 'reject', 'It underfits: both train and CV AP fall.'),
    ('E11', 'Random forest, 200 trees, default settings', 0.290, 0.014, None, 'reject', 'Its trees memorise (train AP 1.000); below E03.'),
    ('E12', 'Random forest, 200 trees, min_samples_leaf=20', 0.325, 0.018, 0.382, 'reject', 'A tie with E03 (higher on 3 folds of 5), and 2,000 times bigger as a file.'),
    ('E13', 'Gradient boosting, default settings', 0.316, 0.010, 0.369, 'reject', 'Lower than E03 on all 5 folds.'),
    ('E14', 'Gradient boosting, max_depth=3, learning_rate=0.05', 0.327, 0.016, 0.374, 'reject', 'A tie with E03; more complex and harder to explain.'),
]
for row in EARLIER_ROWS:
    log_experiment(*row)
show_log()

The next cell adds the rows of the earlier lessons of Module 6 (E15 to E18). If you keep your own log in the local project, skip this cell.

> **Check.** You should see the log with 18 rows, from E01 to E18.

In [ ]:
# The rows of the earlier lessons of Module 6 (id, change, cv_ap, cv_std, valid_ap, decision, reason).
M06_ROWS = [
    ('E15', 'E03 + prior_escalations_90d (11 features)', 0.339, 0.016, 0.391, 'keep', 'Known at creation, and higher than E03 on all 5 folds.'),
    ('E16', 'E03 + order_value_band', 0.328, 0.014, 0.358, 'reject', 'It repeats order_value; the gain is smaller than the noise.'),
    ('E17', 'E03 + customer_escalation_rate', 0.445, 0.015, 0.485, 'reject', 'Leak: computed over later targets, so not known at creation.'),
    ('E18', 'E03 + priority_now', 0.851, 0.024, 0.873, 'reject', 'Leak: the help desk sets it to 1 when a ticket escalates.'),
]
for row in M06_ROWS:
    log_experiment(*row)
show_log()

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Parameters and hyperparameters

An oven does not learn its temperature: you set it before you bake, then you taste the cake. A model has settings like that too. You met them in [Control complexity](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/), where you chose the depth of a tree with CV.

- **Parameters** are the numbers that `fit()` learns from the data, such as the coefficients of logistic regression.
- **Hyperparameters** are the settings that you choose **before** `fit()`, such as `C` of logistic regression, or the depth of a tree. The fit does not change them.

> **Predict.** How many coefficients will the final model learn from the 11 features? (The 4 categories become one column per category.) Then run the cell.

In [ ]:
from ticket_model import FEATURES, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])
model = pipeline.named_steps["model"]
print("parameters learned by fit():", model.coef_.size, "coefficients and 1 intercept")
print("hyperparameters chosen before:", {name: model.get_params()[name] for name in ("C", "max_iter", "class_weight")})

> **Check.** You should see `28 coefficients and 1 intercept`, and `{'C': 1.0, 'max_iter': 1000, 'class_weight': None}`.

The 28 coefficients came from the train tickets. `C = 1.0` came from nobody: it is scikit-learn's default, a guess that works on many datasets. **Tuning** means choosing hyperparameters with data, instead of trusting the guess.

## 2. Score a setting with CV on train

You cannot score a setting on train (a model that memorises wins) or on test (the test set is used once, at the end). So you score it with **cross-validation on train**: fit on 4 folds, score the fifth, 5 times. The course always uses `StratifiedKFold(5, shuffle=True, random_state=0)`, which keeps the same share of escalations in every fold.

Run the cell. It fits the final pipeline 5 times.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(5, shuffle=True, random_state=0)
fold_scores = cross_val_score(build_pipeline(), train[FEATURES], train[TARGET], cv=cv, scoring="average_precision")
print("AP per fold:", fold_scores.round(3))
print(f"mean {fold_scores.mean():.4f}, std {fold_scores.std():.4f}, standard error {fold_scores.std() / 5 ** 0.5:.4f}")

> **Check.** You should see `AP per fold: [0.325 0.321 0.347 0.364 0.34 ]`, then `mean 0.3393, std 0.0156, standard error 0.0070`.

The same model scores from 0.321 to 0.364, only because the folds hold different tickets. The **standard error** of the mean (std ÷ √5 = 0.0070) is the typical size of the chance error in the mean. Two settings whose means differ by less than one standard error are a tie.

## 3. Why the validation month still matters

CV mixes tickets from all 22 months of train, all from before the warranty policy of 2026-05-01. The validation set is May 2026, after it.

> **Predict.** The model's mean score is its average guess of the escalation rate. Will it match the real rate on train? On May? Then run the cell.

In [ ]:
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(build_pipeline(), train[FEATURES], train[TARGET], cv=cv, method="predict_proba")[:, 1]
scores = pipeline.predict_proba(valid[FEATURES])[:, 1]
print(f"train, out of fold: mean score {oof.mean():.3f}, escalation rate {train[TARGET].mean():.3f}")
print(f"valid (May 2026):   mean score {scores.mean():.3f}, escalation rate {valid[TARGET].mean():.3f}")
monthly = train.groupby(train["created_at"].dt.strftime("%Y-%m"))[TARGET].mean()
print(f"train months: escalation rate from {monthly.min():.3f} to {monthly.max():.3f}")

> **Check.** You should see `mean score 0.117, escalation rate 0.117` on train, `mean score 0.120, escalation rate 0.155` on valid, and train months from `0.095` to `0.139`.

`cross_val_predict` gives each train ticket a score from the fold model that did not see it: an **out-of-fold** score. In May, 15.5% of tickets escalate, more than in any train month, and CV cannot show it. So **CV on train chooses the settings, and the validation month checks the chosen model** in the new conditions.

## 4. Grid search: ten settings, the same CV

A **grid search** tries every combination of a short list of values. `GridSearchCV` runs the same CV for each one and ranks them. Priya's grid has 4 logistic regressions and 6 gradient-boosting models: 10 settings × 5 folds = 50 fits.

> **Predict.** Which setting will rank first? Write it down. Then run the cell. It takes about 15 seconds.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

grid = [
    {"model": [LogisticRegression(max_iter=1000)], "model__C": [0.01, 0.1, 1, 10]},
    {"model": [HistGradientBoostingClassifier(random_state=0)],
     "model__max_depth": [2, 3, None], "model__learning_rate": [0.05, 0.1]},
]
search = GridSearchCV(build_pipeline(), grid, cv=cv, scoring="average_precision", n_jobs=-1)
search.fit(train[FEATURES], train[TARGET])
print("settings:", len(search.cv_results_["params"]))

> **Check.** You should see `settings: 10`.

The next cell turns the results into a table, best first. `describe()` gives each setting a short name. Run it.

In [ ]:
def describe(params):
    """A short name for one setting of the grid."""
    if isinstance(params["model"], LogisticRegression):
        return f"logistic, C={params['model__C']}"
    return f"boosting, rate {params['model__learning_rate']}, depth {params['model__max_depth']}"

results = pd.DataFrame(search.cv_results_)
results["setting"] = results["params"].map(describe)
results["se"] = results["std_test_score"] / 5 ** 0.5
table = results[["setting", "mean_test_score", "std_test_score", "se", "rank_test_score"]].sort_values("rank_test_score")
print(table.round(4).to_string(index=False))

> **Check.** The first row is `boosting, rate 0.1, depth 2` with 0.3424 and a standard error of 0.0080. Then `logistic, C=10` 0.3394, `logistic, C=1` 0.3393 and `logistic, C=0.1` 0.3391. The two boosting settings with depth `None` are last (0.3212 and 0.3188).

## 5. The one-standard-error rule

The best setting wins by 0.0031 over logistic regression with C = 1. Is that a real difference? The **one-standard-error rule**: take the best mean, subtract its standard error, and call every setting at or above that limit a tie with the best. From the tied settings, choose the simplest one.

> **Your turn.** Compute `limit` (the best mean minus its standard error), then `tied`: the list of settings in `table` whose mean is at or above `limit`. Run the cell, then the check cell.

In [ ]:
best = table.iloc[0]
limit = ...  # the best mean minus its standard error
tied = ...   # list of the settings (names) with a mean at or above limit
print(limit, tied)

In [ ]:
expect_hash('the tied settings', sorted(tied) if isinstance(tied, list) else tied, '16834725306e3ffa')

Eight settings tie with the best: all four logistic regressions and four boosting models. Priya chooses **logistic regression with the default C = 1**: it ties with the best, it is the simplest, its folds vary less, and its coefficients can be explained to Grace. All four values of C are within 0.002: on this data, tuning C does not matter.

Now confirm the choice once on the validation month, next to the setting that ranked first. Run the cell.

In [ ]:
from sklearn.metrics import average_precision_score

best_boosting = build_pipeline(HistGradientBoostingClassifier(learning_rate=0.1, max_depth=2, random_state=0))
best_boosting.fit(train[FEATURES], train[TARGET])
for name, fitted in [("boosting, rate 0.1, depth 2", best_boosting), ("logistic, C=1", pipeline)]:
    print(f"{name:28s} valid AP {average_precision_score(valid[TARGET], fitted.predict_proba(valid[FEATURES])[:, 1]):.3f}")

> **Check.** You should see `0.395` for boosting and `0.391` for logistic regression.

The difference on May is 0.004, far inside May's chance range of about 0.03. The decision stays.

## 6. Random search: the same budget, other values

A grid of 3 values for each of 3 hyperparameters already has 27 settings, and 5 values each would be 125. A **random search** tries a fixed number of random combinations instead. Often only one or two hyperparameters matter much, and random settings try many different values of each one.

> **Predict.** With 10 random boosting settings, will the best one beat logistic regression by more than one standard error? Then run the cell. It takes about 15 seconds.

In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

space = {"model__learning_rate": loguniform(0.01, 0.3), "model__max_depth": [2, 3, 4, 6, None],
         "model__min_samples_leaf": [20, 50, 100, 200]}
random_search = RandomizedSearchCV(build_pipeline(HistGradientBoostingClassifier(random_state=0)), space,
                                   n_iter=10, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1)
random_search.fit(train[FEATURES], train[TARGET])
found = random_search.cv_results_
for i in found["rank_test_score"].argsort():
    p = found["params"][i]
    print(f"rate {p['model__learning_rate']:.3f}  depth {p['model__max_depth']!s:4}  min leaf {p['model__min_samples_leaf']:3}"
          f"   CV AP {found['mean_test_score'][i]:.4f}  se {found['std_test_score'][i] / 5 ** 0.5:.4f}")

> **Check.** The first line is `rate 0.208  depth 2     min leaf  20   CV AP 0.3441  se 0.0077`. The last line has depth 6 and 0.3302.

The best random setting (0.3441) is a little higher than the best of the grid (0.3424). But the limit is 0.3441 − 0.0077 = 0.3364, and logistic regression with C = 1 (0.3393) is above it: still a tie. The decision does not change.

## 7. Write it in the experiment log

The setup wrote your log so far, E01 to E18. The next cell adds three rows: the best setting of the grid (E19), the best value of `C` (E20) and the best random setting (E21). All three are rejects: each ties with E15. Run it.

In [ ]:
best = table.iloc[0]
c_10 = table.set_index("setting").loc["logistic, C=10"]
top = found["rank_test_score"].argmin()
log_experiment("E19", "Grid search best: gradient boosting, learning_rate=0.1, max_depth=2",
               best["mean_test_score"], best["std_test_score"], 0.395, "reject",
               "Within one standard error (0.008) of E15, and more complex.")
log_experiment("E20", "E15 with C=10 (the best C; C=0.01 to 10 are within 0.002)",
               c_10["mean_test_score"], c_10["std_test_score"], None, "reject",
               "No real difference from C=1, so keep the default.")
log_experiment("E21", "Random search best: gradient boosting, learning_rate=0.208, max_depth=2, min_samples_leaf=20",
               found["mean_test_score"][top], found["std_test_score"][top], None, "reject",
               "Less than one standard error (0.0077) above E15.")
show_log()

> **Check.** You should see 21 rows. E19 (`0.342`, `0.018`, `0.395`), E20 (`0.339`, `0.016`) and E21 (`0.344`, `0.017`) are `reject`. E15 stays the row to keep.

A search that changes nothing is still worth a row: it is the evidence that the simple model is not a lazy choice.

## 8. Misconception: tune on the test set

Tomás says: "We have a test set. Let us try the 10 settings on it and keep the best." What would that do? To keep the real test set clean, Priya simulates it inside May. She cuts the May tickets at random into two halves. Half A plays the test set that Tomás tunes on. Half B plays new tickets. She repeats it 200 times.

The next cell fits the 10 grid settings on train and scores May. Run it. It takes about 10 seconds.

In [ ]:
from sklearn.base import clone

may_scores = {}
for params in search.cv_results_["params"]:
    settings = {key.split("__")[1]: value for key, value in params.items() if "__" in key}
    model = clone(params["model"]).set_params(**settings)
    fitted = build_pipeline(model).fit(train[FEATURES], train[TARGET])
    may_scores[describe(params)] = fitted.predict_proba(valid[FEATURES])[:, 1]
print(len(may_scores), "settings scored")

> **Check.** You should see `10 settings scored`.

> **Predict.** Tomás reports the AP of the best setting on half A. Will that setting score as well on half B? Then run the cell.

In [ ]:
truth = valid[TARGET].to_numpy()
rng = np.random.default_rng(0)
reported, on_new, fixed = [], [], []
for draw in range(200):
    order = rng.permutation(len(truth))
    half_a, half_b = order[:589], order[589:]
    ap_a = {name: average_precision_score(truth[half_a], s[half_a]) for name, s in may_scores.items()}
    chosen = max(ap_a, key=ap_a.get)
    reported.append(ap_a[chosen])
    on_new.append(average_precision_score(truth[half_b], may_scores[chosen][half_b]))
    fixed.append(average_precision_score(truth[half_b], may_scores["logistic, C=1"][half_b]))
print(f"reported (best of 10 on half A): {np.mean(reported):.3f}")
print(f"the same setting on half B:      {np.mean(on_new):.3f}")
print(f"logistic, C=1 on half B:         {np.mean(fixed):.3f}")

> **Check.** You should see `reported (best of 10 on half A): 0.411`, `the same setting on half B: 0.389` and `logistic, C=1 on half B: 0.399`.

The best of 10 on half A looks 0.022 better than it is on new tickets: about three CV standard errors. And the setting chosen this way does no better on new tickets than logistic regression with C = 1, chosen in advance. **When you choose on a set, its score is no longer an honest estimate.** That is why the test set is used once, after every choice (Lesson 3).

## 9. Change one thing: rank by ROC AUC

> **Your turn.** In the next cell, change `scoring_name` to `"roc_auc"`, and run the cell (about 15 seconds). Then answer: is `logistic, C=1` within one standard error of the best setting? Set `within` to `True` or `False`, run the cell again, and run the check cell.

In [ ]:
scoring_name = "average_precision"  # change this to the name of the ROC AUC scoring
search_auc = GridSearchCV(build_pipeline(), grid, cv=cv, scoring=scoring_name, n_jobs=-1)
search_auc.fit(train[FEATURES], train[TARGET])
auc = pd.DataFrame(search_auc.cv_results_)
auc["setting"] = auc["params"].map(describe)
auc["se"] = auc["std_test_score"] / 5 ** 0.5
print(auc[["setting", "mean_test_score", "se", "rank_test_score"]].sort_values("rank_test_score").head(4).round(4).to_string(index=False))
within = ...  # True or False

In [ ]:
expect_hash('your scoring and answer', (scoring_name, within), 'ccd623affeff63dc')

With ROC AUC, the same boosting setting is first (0.7719, standard error 0.0026), and `logistic, C=1` is 0.7696: 0.0023 lower, less than one standard error. The decision does not change. A different metric can change the ranking, so choose the metric before the search, not after.

## Next

You tuned the model with CV on train, decided with the one-standard-error rule, and kept the test set sealed. Lesson 3 turns the scores into flags for Grace's team, checks whether the scores can be read as chances, and then opens the test set once. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.